# EXACT 2026 — QLoRA SFT of Qwen2.5-7B-Instruct (free Colab/Kaggle T4)

Produces a LoRA adapter that teaches the model **stable JSON output + grounded
explanation phrasing + cot/premises format**. It does NOT teach the model to
compute — the deterministic solver still does the math (see ADR 0001/0006).

**Why Qwen2.5-7B-Instruct** (not Qwen3-8B): ADR 0008 — Qwen3 forces a hidden
reasoning channel that breaks utility extraction. Qwen2.5-Instruct emits clean
direct JSON. Both are open-source and ≤8B (competition-legal).

**Runtime**: set Colab/Kaggle accelerator to **GPU (T4, 16 GB)**.
Free T4 QLoRA-trains 7B comfortably (~30-60 min for 2 epochs on ~1.8k rows).

**Inputs**: upload `sft_chat_train.jsonl` and `sft_chat_val.jsonl` from the
repo's `data/processed/` (produced locally by `python -m exact_agent.train.prepare_sft`).

**Output**: `exact_qwen25_7b_lora.zip` — download it, then see
`docs/sft_colab_guide.md` for how to serve it.

## 1. Install — Unsloth only (it self-pins a consistent stack)

**Use a CLEAN runtime.** If you already hit errors in this session:
Runtime → **Disconnect and delete runtime**, reopen the notebook, then
Run all. A runtime polluted by earlier pip attempts has half-downgraded
packages and will NOT self-heal — only a brand-new runtime gives the
consistent stack `pip install unsloth` provides. Do **not** add any
other pip line (extra `trl`/`transformers` pins are what broke prior
runs).

In [ ]:
%%capture
# ONE install, nothing else. Unsloth's release pins a mutually
# consistent torch/transformers/tokenizers/trl/peft/bnb set tested on
# current Colab. Adding our own pins (esp. with --no-deps) is what
# created the tokenizers / fix_untrained_tokens conflicts.
!pip install -q unsloth

## 2. Upload the prepared SFT data

Run this cell, then pick `sft_chat_train.jsonl` **and** `sft_chat_val.jsonl`
from your machine (`data/processed/` in the repo). On Kaggle, add them as a
Dataset instead and set the two paths.

In [ ]:
import os
TRAIN_PATH, VAL_PATH = "sft_chat_train.jsonl", "sft_chat_val.jsonl"
if not (os.path.exists(TRAIN_PATH) and os.path.exists(VAL_PATH)):
    try:
        from google.colab import files  # Colab
        print("Select sft_chat_train.jsonl and sft_chat_val.jsonl ...")
        files.upload()
    except ImportError:
        raise SystemExit(
            "Not on Colab. Add the two JSONL files as a Kaggle dataset and "
            "set TRAIN_PATH / VAL_PATH to their paths."
        )
assert os.path.exists(TRAIN_PATH) and os.path.exists(VAL_PATH), "missing data files"
print("data ok:", os.path.getsize(TRAIN_PATH), os.path.getsize(VAL_PATH), "bytes")

## 3. Load Qwen2.5-7B-Instruct in 4-bit

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 4. Build the dataset (Qwen chat template, train on the assistant turn)

Each row already has `messages = [system, user, assistant]` (from
`prepare_sft.py`). We render with the model's chat template; the assistant
turn is the JSON envelope the API also emits.

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": TRAIN_PATH, "validation": VAL_PATH})

def format_row(ex):
    text = tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

ds = ds.map(format_row, remove_columns=ds["train"].column_names)
print(ds)
print("--- sample ---\n", ds["train"][0]["text"][:600])

## 5. Train (hyperparams mirror configs/training/sft_qwen3_8b.yaml)

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig

# trl/transformers renamed Trainer's `tokenizer` -> `processing_class`
# at different times. Pick whichever the installed SFTTrainer accepts
# so this cell works regardless of which stack `pip install unsloth`
# pinned today.
_params = inspect.signature(SFTTrainer.__init__).parameters
_tok_kw = "processing_class" if "processing_class" in _params else "tokenizer"

trainer = SFTTrainer(
    model=model,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        output_dir="outputs",
        num_train_epochs=2,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        bf16=False,
        fp16=True,
        seed=42,
        report_to="none",
    ),
    **{_tok_kw: tokenizer},
)
trainer.train()

## 6. Save the LoRA adapter + download

In [ ]:
ADAPTER_DIR = "exact_qwen25_7b_lora"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
!zip -qr exact_qwen25_7b_lora.zip {ADAPTER_DIR}
print("adapter files:", os.listdir(ADAPTER_DIR))
try:
    from google.colab import files
    files.download("exact_qwen25_7b_lora.zip")
except ImportError:
    print("Kaggle: the zip is in the working dir — add it to Output and download.")

## 7. (Optional) quick sanity generation

Confirm the adapter emits a clean JSON envelope before you download.

In [ ]:
FastLanguageModel.for_inference(model)
msgs = [
    {"role": "system", "content": "You are EXACT 2026 — return ONLY a JSON object with keys answer, explanation."},
    {"role": "user", "content": "Question: A capacitor C=100 uF at U=30 V. Energy?"},
]
inp = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=inp, max_new_tokens=200, temperature=0.2, do_sample=True)
print(tokenizer.decode(out[0][inp.shape[1]:], skip_special_tokens=True))